## Importaciones y Configuracion Visual

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# Configuración de estilo para las gráficas
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['savefig.dpi'] = 300 # Alta resolución para el documento
plt.rcParams['savefig.bbox'] = 'tight'

# Definición de nuestras variables de estudio
escenarios = ['none', 'auth', 'encrypt', 'access']
payloads = ['256', '1024', '16384']

## Extracción y Limpieza de Latencia

In [ ]:
print("Procesando datos de Latencia...")
datos_latencia = []

for escenario in escenarios:
    for payload in payloads:
        # Buscar todas las corridas para esta combinación
        archivos = glob.glob(f"resultados_latencia/Latencia_{escenario}_{payload}B_run*.csv")
        
        for archivo in archivos:
            # Leer asumiendo que solo hay 3 columnas de datos
            try:
                df_temp = pd.read_csv(archivo, names=['SeqNum', 'PayloadSize', 'Latency'], on_bad_lines='skip')
                
                # FORZAR A NÚMEROS: Esto convierte los textos (como "[Subscriber]...") en NaN
                df_temp['SeqNum'] = pd.to_numeric(df_temp['SeqNum'], errors='coerce')
                df_temp['Latency'] = pd.to_numeric(df_temp['Latency'], errors='coerce')
                
                # Eliminar las filas que se volvieron NaN (los textos)
                df_temp = df_temp.dropna()
                
                # Descartar el calentamiento (primeras 1000 muestras válidas)
                df_limpio = df_temp.iloc[1000:].copy() 
                
                # Etiquetar los datos
                df_limpio['Escenario'] = escenario
                df_limpio['Payload'] = int(payload)
                
                datos_latencia.append(df_limpio)
            except Exception as e:
                print(f"Error procesando {archivo}: {e}")

# Unir todo en un mega DataFrame
df_lat = pd.concat(datos_latencia, ignore_index=True)
# Convertir latencia a milisegundos (ms) para gráficas más legibles, o dejar en us
df_lat['Latency_ms'] = df_lat['Latency'] / 1000.0

print(f"¡Listo! Se cargaron {len(df_lat):,} muestras válidas de latencia.")

## Extracción de Recursos (CPU y RAM)

In [ ]:
print("Procesando datos de Recursos (CPU/RAM)...")
datos_recursos = []

for escenario in escenarios:
    for payload in payloads:
        archivos = glob.glob(f"resultados_recursos_pi/Recursos_{escenario}_{payload}B_run*.csv")
        
        for archivo in archivos:
            try:
                # El script bash sí le ponía header: Timestamp,CPU(%),RAM(MB)
                df_temp = pd.read_csv(archivo)
                df_temp['Escenario'] = escenario
                df_temp['Payload'] = int(payload)
                datos_recursos.append(df_temp)
            except Exception as e:
                pass # Si un archivo está vacío por error en alguna corrida, lo ignora

df_rec = pd.concat(datos_recursos, ignore_index=True)
df_rec.rename(columns={'CPU(%)': 'CPU', 'RAM(MB)': 'RAM'}, inplace=True)
print(f"¡Listo! Se cargaron {len(df_rec):,} registros de monitoreo.")

## Estadísticas Descriptivas y Cálculo de Overhead

In [ ]:
# 1. Agrupar y calcular estadísticas de latencia
stats_lat = df_lat.groupby(['Payload', 'Escenario'])['Latency_ms'].agg(
    Media='mean',
    Desv_Estandar='std',
    Mediana='median',
    P99=lambda x: x.quantile(0.99)
).reset_index()

# 2. Reordenar los escenarios lógicamente
orden_escenarios = pd.CategoricalDtype(['none', 'auth', 'encrypt', 'access'], ordered=True)
stats_lat['Escenario'] = stats_lat['Escenario'].astype(orden_escenarios)
stats_lat = stats_lat.sort_values(['Payload', 'Escenario'])

# 3. Calcular Overhead vs 'none'
def calc_overhead(row, df_stats):
    if row['Escenario'] == 'none':
        return 0.0
    
    # Buscar la latencia base (none) para este mismo payload
    lat_base = df_stats[(df_stats['Payload'] == row['Payload']) & 
                        (df_stats['Escenario'] == 'none')]['Media'].values[0]
    
    overhead = ((row['Media'] - lat_base) / lat_base) * 100
    return overhead

stats_lat['Overhead_%'] = stats_lat.apply(lambda row: calc_overhead(row, stats_lat), axis=1)

# Mostrar la tabla final
display(stats_lat.round(2))

## Gráfica 1 - Distribución de Latencia (Boxplots)

In [ ]:
plt.figure(figsize=(12, 6))

ax = sns.boxplot(data=df_lat, x='Payload', y='Latency_ms', hue='Escenario', 
                 palette='viridis', fliersize=2)

plt.yscale('log') # Escala logarítmica (quita esto si la diferencia no es tan extrema)
plt.title('Distribución de Latencia por Escenario de Seguridad y Payload', pad=15, fontweight='bold')
plt.xlabel('Tamaño de Payload (Bytes)', fontweight='bold')
plt.ylabel('Latencia (ms) - Escala Logarítmica', fontweight='bold')
plt.legend(title='Escenario')

plt.tight_layout()
plt.savefig('grafica_boxplot_latencia.png')
plt.show()

## Gráfica 2 - Impacto de la Criptografía en la CPU

In [ ]:
# Ordenar para la gráfica
df_rec['Escenario'] = df_rec['Escenario'].astype(orden_escenarios)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_rec, x='Payload', y='CPU', hue='Escenario', 
            palette='magma', errorbar='sd', capsize=0.1)

plt.title('Consumo Promedio de CPU vs Tamaño de Mensaje', pad=15, fontweight='bold')
plt.xlabel('Tamaño de Payload (Bytes)', fontweight='bold')
plt.ylabel('Uso de CPU (%)', fontweight='bold')
plt.ylim(0, 100) # La CPU va de 0 a 100

plt.legend(title='Escenario')
plt.tight_layout()
plt.savefig('grafica_cpu_impacto.png')
plt.show()

## Gráfica 3 - Escalabilidad de la Latencia

In [ ]:
plt.figure(figsize=(10, 6))

sns.lineplot(data=stats_lat, x='Payload', y='Media', hue='Escenario', 
             marker='o', markersize=10, linewidth=2.5, palette='tab10')

plt.title('Escalabilidad: Latencia Media vs Payload', pad=15, fontweight='bold')
plt.xlabel('Tamaño de Payload (Bytes)', fontweight='bold')
plt.ylabel('Latencia Promedio (ms)', fontweight='bold')
plt.xticks([256, 1024, 16384]) # Forzar las marcas en el eje X

plt.grid(True, which="both", ls="--", alpha=0.6)
plt.legend(title='Escenario')
plt.tight_layout()
plt.savefig('grafica_escalabilidad_lineas.png')
plt.show()